## Ενσωματώσεις

Στο προηγούμενο παράδειγμά μας, δουλέψαμε με διανύσματα bag-of-words υψηλής διάστασης με μήκος `vocab_size`, και μετατρέπαμε ρητά διανύσματα χαμηλής διάστασης με θέση σε αραιή αναπαράσταση one-hot. Αυτή η αναπαράσταση one-hot δεν είναι αποδοτική σε μνήμη, επιπλέον, κάθε λέξη αντιμετωπίζεται ανεξάρτητα από τις άλλες, δηλαδή τα one-hot κωδικοποιημένα διανύσματα δεν εκφράζουν καμία σημασιολογική ομοιότητα μεταξύ λέξεων.

Σε αυτή την ενότητα, θα συνεχίσουμε την εξερεύνηση του συνόλου δεδομένων **News AG**. Για αρχή, ας φορτώσουμε τα δεδομένα και ας πάρουμε μερικούς ορισμούς από το προηγούμενο σημειωματάριο.


In [1]:
import torch
import torchtext
import numpy as np
from torchnlp import *
train_dataset, test_dataset, classes, vocab = load_dataset()
vocab_size = len(vocab)
print("Vocab size = ",vocab_size)

Loading dataset...


d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\train.csv: 29.5MB [00:01, 18.8MB/s]                            
d:\WORK\ai-for-beginners\5-NLP\14-Embeddings\data\test.csv: 1.86MB [00:00, 11.2MB/s]                          


Building vocab...
Vocab size =  95812



## Τι είναι το embedding;

Η ιδέα του **embedding** είναι να αναπαραστήσουμε λέξεις με χαμηλότερης διάστασης πυκνούς διανύσματα, που με κάποιο τρόπο αντικατοπτρίζουν το σημασιολογικό νόημα μιας λέξης. Θα συζητήσουμε αργότερα πώς να δημιουργήσουμε ουσιαστικά word embeddings, αλλά προς το παρόν ας θεωρήσουμε τα embeddings ως έναν τρόπο μείωσης της διάστασης ενός διανύσματος λέξης.

Έτσι, η στρώση embedding θα παίρνει μια λέξη ως είσοδο και θα παράγει ένα διάνυσμα εξόδου με καθορισμένη διάσταση `embedding_size`. Με μια έννοια, είναι πολύ παρόμοια με τη στρώση `Linear`, αλλά αντί να παίρνει διάνυσμα one-hot κωδικοποιημένο, θα μπορεί να πάρει έναν αριθμό λέξης ως είσοδο.

Χρησιμοποιώντας τη στρώση embedding ως πρώτη στρώση στο δίκτυό μας, μπορούμε να περάσουμε από το μοντέλο bag-of-words σε μοντέλο **embedding bag**, όπου πρώτα μετατρέπουμε κάθε λέξη στο κείμενο μας στο αντίστοιχο embedding, και μετά υπολογίζουμε κάποια συνάθροιση όλων αυτών των embeddings, όπως `sum`, `average` ή `max`.

![Εικόνα που δείχνει έναν ταξινομητή embedding για πέντε διαδοχικές λέξεις.](../../../../../translated_images/el/embedding-classifier-example.b77f021a7ee67eee.webp)

Το ταξινομητικό νευρωνικό μας δίκτυο θα ξεκινά με στρώση embedding, μετά στρώση συναθροιστικής λειτουργίας, και γραμμικό ταξινομητή από πάνω της:


In [2]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.Embedding(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, x):
        x = self.embedding(x)
        x = torch.mean(x,dim=1)
        return self.fc(x)

### Διαχείριση μεταβλητού μεγέθους ακολουθίας

Ως αποτέλεσμα αυτής της αρχιτεκτονικής, τα minibatches για το δίκτυό μας θα πρέπει να δημιουργούνται με έναν συγκεκριμένο τρόπο. Στην προηγούμενη ενότητα, όταν χρησιμοποιούσαμε bag-of-words, όλα τα BoW tensors σε ένα minibatch είχαν το ίδιο μέγεθος `vocab_size`, ανεξάρτητα από το πραγματικό μήκος της ακολουθίας κειμένου μας. Μόλις περάσουμε στα embeddings λέξεων, θα καταλήξουμε με μεταβλητό αριθμό λέξεων σε κάθε δείγμα κειμένου, και όταν συνδυάζουμε αυτά τα δείγματα σε minibatches θα πρέπει να εφαρμόσουμε κάποιο padding.

Αυτό μπορεί να γίνει χρησιμοποιώντας την ίδια τεχνική παροχής της συνάρτησης `collate_fn` στην πηγή δεδομένων:


In [3]:
def padify(b):
    # b is the list of tuples of length batch_size
    #   - first element of a tuple = label, 
    #   - second = feature (text sequence)
    # build vectorized sequence
    v = [encode(x[1]) for x in b]
    # first, compute max length of a sequence in this minibatch
    l = max(map(len,v))
    return ( # tuple of two tensors - labels and features
        torch.LongTensor([t[0]-1 for t in b]),
        torch.stack([torch.nn.functional.pad(torch.tensor(t),(0,l-len(t)),mode='constant',value=0) for t in v])
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=padify, shuffle=True)

### Εκπαίδευση ταξινομητή ενσωμάτωσης

Τώρα που έχουμε ορίσει τον κατάλληλο φορτωτή δεδομένων, μπορούμε να εκπαιδεύσουμε το μοντέλο χρησιμοποιώντας τη λειτουργία εκπαίδευσης που έχουμε ορίσει στην προηγούμενη ενότητα:


In [4]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)
train_epoch(net,train_loader, lr=1, epoch_size=25000)

3200: acc=0.6415625
6400: acc=0.6865625
9600: acc=0.7103125
12800: acc=0.726953125
16000: acc=0.739375
19200: acc=0.75046875
22400: acc=0.7572321428571429


(0.889799795315499, 0.7623160588611644)

> **Σημείωση**: Εκπαιδευόμαστε μόνο για 25 χιλιάδες εγγραφές εδώ (λιγότερο από μια πλήρη εποχή) για λόγους χρόνου, αλλά μπορείτε να συνεχίσετε την εκπαίδευση, να γράψετε μια συνάρτηση για εκπαίδευση σε αρκετές εποχές και να πειραματιστείτε με την τιμή του ρυθμού μάθησης για να επιτύχετε υψηλότερη ακρίβεια. Θα πρέπει να μπορείτε να φτάσετε σε ακρίβεια περίπου 90%.


### Στρώμα EmbeddingBag και Αναπαράσταση Ακολουθιών Με Μεταβλητό Μήκος

Στην προηγούμενη αρχιτεκτονική, χρειαζόμασταν να γεμίσουμε με padding όλες τις ακολουθίες στο ίδιο μήκος ώστε να χωρέσουν σε ένα minibatch. Αυτή δεν είναι η πιο αποδοτική μέθοδος για την αναπαράσταση ακολουθιών μεταβλητού μήκους - μια άλλη προσέγγιση θα ήταν η χρήση διανύσματος **offset**, που θα κρατά τις μετατοπίσεις όλων των ακολουθιών αποθηκευμένων σε ένα μεγάλο διάνυσμα.

![Εικόνα που δείχνει μια αναπαράσταση ακολουθίας με offset](../../../../../translated_images/el/offset-sequence-representation.eb73fcefb29b46ee.webp)

> **Σημείωση**: Στην παραπάνω εικόνα, δείχνουμε μια ακολουθία χαρακτήρων, αλλά στο παράδειγμά μας δουλεύουμε με ακολουθίες λέξεων. Ωστόσο, η γενική αρχή της αναπαράστασης ακολουθιών με διάνυσμα offset παραμένει η ίδια.

Για να δουλέψουμε με την αναπαράσταση offset, χρησιμοποιούμε το στρώμα [`EmbeddingBag`](https://pytorch.org/docs/stable/generated/torch.nn.EmbeddingBag.html). Είναι παρόμοιο με το `Embedding`, αλλά παίρνει διάνυσμα περιεχομένου και διάνυσμα offset ως είσοδο, και περιλαμβάνει επίσης στρώμα μέσου όρου, το οποίο μπορεί να είναι `mean`, `sum` ή `max`.

Εδώ είναι ένα τροποποιημένο δίκτυο που χρησιμοποιεί `EmbeddingBag`:


In [5]:
class EmbedClassifier(torch.nn.Module):
    def __init__(self, vocab_size, embed_dim, num_class):
        super().__init__()
        self.embedding = torch.nn.EmbeddingBag(vocab_size, embed_dim)
        self.fc = torch.nn.Linear(embed_dim, num_class)

    def forward(self, text, off):
        x = self.embedding(text, off)
        return self.fc(x)

Για να προετοιμάσουμε το σύνολο δεδομένων για εκπαίδευση, πρέπει να παρέχουμε μια συνάρτηση μετατροπής που θα προετοιμάσει το διάνυσμα μετατόπισης:


In [6]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1])) for t in b]
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)

Σημειώστε, ότι σε αντίθεση με όλα τα προηγούμενα παραδείγματα, το δίκτυό μας τώρα δέχεται δύο παραμέτρους: διάνυσμα δεδομένων και διάνυσμα μετατόπισης, τα οποία έχουν διαφορετικά μεγέθη. Ομοίως, ο φορτωτής δεδομένων μας παρέχει επίσης 3 τιμές αντί για 2: τόσο τα διανύσματα κειμένου όσο και τα διανύσματα μετατόπισης παρέχονται ως χαρακτηριστικά. Επομένως, πρέπει να προσαρμόσουμε ελαφρώς τη λειτουργία εκπαίδευσής μας για να το λάβουμε υπόψη:


In [7]:
net = EmbedClassifier(vocab_size,32,len(classes)).to(device)

def train_epoch_emb(net,dataloader,lr=0.01,optimizer=None,loss_fn = torch.nn.CrossEntropyLoss(),epoch_size=None, report_freq=200):
    optimizer = optimizer or torch.optim.Adam(net.parameters(),lr=lr)
    loss_fn = loss_fn.to(device)
    net.train()
    total_loss,acc,count,i = 0,0,0,0
    for labels,text,off in dataloader:
        optimizer.zero_grad()
        labels,text,off = labels.to(device), text.to(device), off.to(device)
        out = net(text, off)
        loss = loss_fn(out,labels) #cross_entropy(out,labels)
        loss.backward()
        optimizer.step()
        total_loss+=loss
        _,predicted = torch.max(out,1)
        acc+=(predicted==labels).sum()
        count+=len(labels)
        i+=1
        if i%report_freq==0:
            print(f"{count}: acc={acc.item()/count}")
        if epoch_size and count>epoch_size:
            break
    return total_loss.item()/count, acc.item()/count


train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6153125
6400: acc=0.6615625
9600: acc=0.6932291666666667
12800: acc=0.715078125
16000: acc=0.7270625
19200: acc=0.7382291666666667
22400: acc=0.7486160714285715


(22.771553103007037, 0.7551983365323096)


## Σημασιολογικές Ενσωματώσεις: Word2Vec

Στο προηγούμενο παράδειγμά μας, το στρώμα ενσωμάτωσης του μοντέλου έμαθε να αντιστοιχίζει λέξεις σε αναπαράσταση διανύσματος, ωστόσο, αυτή η αναπαράσταση δεν είχε μεγάλη σημασιολογική αξία. Θα ήταν καλό να μάθουμε μια τέτοια αναπαράσταση διανύσματος, όπου παρόμοιες λέξεις ή συνώνυμα να αντιστοιχούν σε διανύσματα που είναι κοντά το ένα στο άλλο ως προς κάποια μέτρηση απόστασης διανυσμάτων (π.χ. Ευκλείδεια απόσταση).

Για να το πετύχουμε αυτό, πρέπει να προ-εκπαιδεύσουμε το μοντέλο ενσωμάτωσης μας σε μια μεγάλη συλλογή κειμένων με συγκεκριμένο τρόπο. Μια από τις πρώτες μεθόδους εκπαίδευσης σημασιολογικών ενσωματώσεων ονομάζεται [Word2Vec](https://en.wikipedia.org/wiki/Word2vec). Βασίζεται σε δύο βασικές αρχιτεκτονικές που χρησιμοποιούνται για την παραγωγή κατανεμημένης αναπαράστασης των λέξεων:

 - **Συνεχής τσάντα-λέξεων** (CBoW) — σε αυτή την αρχιτεκτονική, εκπαιδεύουμε το μοντέλο να προβλέπει μια λέξη από το περιβάλλον του. Δεδομένου του ngram $(W_{-2},W_{-1},W_0,W_1,W_2)$, ο στόχος του μοντέλου είναι να προβλέψει το $W_0$ από το $(W_{-2},W_{-1},W_1,W_2)$.
 - **Συνεχής skip-gram** είναι αντίθετο του CBoW. Το μοντέλο χρησιμοποιεί το περιβάλλον των λέξεων γειτονιάς για να προβλέψει την τρέχουσα λέξη.

Το CBoW είναι ταχύτερο, ενώ το skip-gram είναι πιο αργό, αλλά κάνει καλύτερη δουλειά στην αναπαράσταση των λέξεων που εμφανίζονται πιο σπάνια.

![Εικόνα που δείχνει και τους αλγόριθμους CBoW και Skip-Gram για τη μετατροπή λέξεων σε διανύσματα.](../../../../../translated_images/el/example-algorithms-for-converting-words-to-vectors.fbe9207a726922f6.webp)

Για να πειραματιστούμε με το word2vec embedding προ-εκπαιδευμένο στο σύνολο δεδομένων Google News, μπορούμε να χρησιμοποιήσουμε τη βιβλιοθήκη **gensim**. Παρακάτω βρίσκουμε τις λέξεις που είναι πιο παρόμοιες με το 'neural'

> **Σημείωση:** Όταν δημιουργείς για πρώτη φορά διανύσματα λέξεων, η λήψη τους μπορεί να πάρει κάποιο χρόνο!


In [8]:
import gensim.downloader as api
w2v = api.load('word2vec-google-news-300')

In [9]:
for w,p in w2v.most_similar('neural'):
    print(f"{w} -> {p}")

neuronal -> 0.7804799675941467
neurons -> 0.7326500415802002
neural_circuits -> 0.7252851724624634
neuron -> 0.7174385190010071
cortical -> 0.6941086649894714
brain_circuitry -> 0.6923246383666992
synaptic -> 0.6699118614196777
neural_circuitry -> 0.6638563275337219
neurochemical -> 0.6555314064025879
neuronal_activity -> 0.6531826257705688


Μπορούμε επίσης να υπολογίσουμε ενσωματώσεις διανυσμάτων από τη λέξη, για να χρησιμοποιηθούν στην εκπαίδευση του μοντέλου ταξινόμησης (δείχνουμε μόνο τα πρώτα 20 στοιχεία του διανύσματος για σαφήνεια):


In [10]:
w2v.word_vec('play')[:20]

array([ 0.01226807,  0.06225586,  0.10693359,  0.05810547,  0.23828125,
        0.03686523,  0.05151367, -0.20703125,  0.01989746,  0.10058594,
       -0.03759766, -0.1015625 , -0.15820312, -0.08105469, -0.0390625 ,
       -0.05053711,  0.16015625,  0.2578125 ,  0.10058594, -0.25976562],
      dtype=float32)

Το υπέροχο με τις σημασιολογικές ενσωματώσεις είναι ότι μπορείτε να χειριστείτε τον κωδικοποιητή διανυσμάτων για να αλλάξετε τη σημασιολογία. Για παράδειγμα, μπορούμε να ζητήσουμε να βρούμε μια λέξη, της οποίας η αναπαράσταση διανύσματος θα ήταν όσο το δυνατόν πιο κοντά στις λέξεις *king* και *woman*, και όσο το δυνατόν πιο μακριά από τη λέξη *man*:


In [10]:
w2v.most_similar(positive=['king','woman'],negative=['man'])[0]

('queen', 0.7118192911148071)

Τόσο το CBoW όσο και τα Skip-Grams είναι “προβλεπτικές” ενσωματώσεις, αφού λαμβάνουν υπόψη μόνο τοπικά συμφραζόμενα. Το Word2Vec δεν εκμεταλλεύεται το παγκόσμιο πλαίσιο. 

**FastText** βασίζεται στο Word2Vec μαθαίνοντας διανυσματικές αναπαραστάσεις για κάθε λέξη και τα χαρακτήρων n-γράμματα που βρίσκονται μέσα σε κάθε λέξη. Οι τιμές των αναπαραστάσεων στη συνέχεια μέσονται σε ένα διάνυσμα σε κάθε βήμα εκπαίδευσης. Παρόλο που αυτό προσθέτει πολλούς επιπλέον υπολογισμούς στην προκαταρκτική εκπαίδευση, επιτρέπει στις ενσωματώσεις λέξεων να κωδικοποιούν πληροφορίες υπολέξεων. 

Μια άλλη μέθοδος, **GloVe**, αξιοποιεί την ιδέα του πίνακα συνύπαρξης, χρησιμοποιώντας νευρωνικές μεθόδους για να αποσυνθέσει τον πίνακα συνύπαρξης σε πιο εκφραστικά και μη γραμμικά διανύσματα λέξεων.

Μπορείτε να πειραματιστείτε με το παράδειγμα αλλάζοντας τις ενσωματώσεις σε FastText και GloVe, καθώς το gensim υποστηρίζει αρκετά διαφορετικά μοντέλα ενσωμάτωσης λέξεων.


## Χρήση Προεκπαιδευμένων Ενσωματώσεων στο PyTorch

Μπορούμε να τροποποιήσουμε το παραπάνω παράδειγμα ώστε να προγεμίσουμε τον πίνακα στο επίπεδο ενσωμάτωσής μας με σημασιολογικές ενσωματώσεις, όπως το Word2Vec. Πρέπει να λάβουμε υπόψη ότι τα λεξιλόγια της προεκπαιδευμένης ενσωμάτωσης και του σώματος κειμένου μας πιθανότατα δεν θα ταιριάζουν, οπότε θα αρχικοποιήσουμε τα βάρη για τις ελλείπουσες λέξεις με τυχαίες τιμές:


In [11]:
embed_size = len(w2v.get_vector('hello'))
print(f'Embedding size: {embed_size}')

net = EmbedClassifier(vocab_size,embed_size,len(classes))

print('Populating matrix, this will take some time...',end='')
found, not_found = 0,0
for i,w in enumerate(vocab.get_itos()):
    try:
        net.embedding.weight[i].data = torch.tensor(w2v.get_vector(w))
        found+=1
    except:
        net.embedding.weight[i].data = torch.normal(0.0,1.0,(embed_size,))
        not_found+=1

print(f"Done, found {found} words, {not_found} words missing")
net = net.to(device)

Embedding size: 300
Populating matrix, this will take some time...Done, found 41080 words, 54732 words missing


Τώρα ας εκπαιδεύσουμε το μοντέλο μας. Σημειώστε ότι ο χρόνος που απαιτείται για την εκπαίδευση του μοντέλου είναι σημαντικά μεγαλύτερος από το προηγούμενο παράδειγμα, λόγω του μεγαλύτερου μεγέθους του embedding layer, και του πολύ μεγαλύτερου αριθμού παραμέτρων. Επίσης, εξαιτίας αυτού, ίσως χρειαστεί να εκπαιδεύσουμε το μοντέλο μας σε περισσότερα παραδείγματα αν θέλουμε να αποφύγουμε την υπερπροσαρμογή.


In [12]:
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6359375
6400: acc=0.68109375
9600: acc=0.7067708333333333
12800: acc=0.723671875
16000: acc=0.73625
19200: acc=0.7463541666666667
22400: acc=0.7560714285714286


(214.1013875559821, 0.7626759436980166)

Στη δική μας περίπτωση δεν παρατηρούμε μεγάλη αύξηση στην ακρίβεια, γεγονός που πιθανόν οφείλεται σε αρκετά διαφορετικά λεξιλόγια. 
Για να ξεπεράσουμε το πρόβλημα των διαφορετικών λεξιλογίων, μπορούμε να χρησιμοποιήσουμε μία από τις ακόλουθες λύσεις:
* Επαναεκπαίδευση μοντέλου word2vec στο δικό μας λεξιλόγιο
* Φόρτωση του συνόλου δεδομένων μας με το λεξιλόγιο από το προεκπαιδευμένο μοντέλο word2vec. Το λεξιλόγιο που χρησιμοποιείται για τη φόρτωση του συνόλου δεδομένων μπορεί να οριστεί κατά τη φόρτωση.

Η δεύτερη προσέγγιση φαίνεται πιο εύκολη, ειδικά επειδή το πλαίσιο PyTorch `torchtext` περιλαμβάνει ενσωματωμένη υποστήριξη για embeddings. Μπορούμε, για παράδειγμα, να δημιουργήσουμε ένα λεξιλόγιο βασισμένο σε GloVe με τον ακόλουθο τρόπο:


In [14]:
vocab = torchtext.vocab.GloVe(name='6B', dim=50)

100%|█████████▉| 399999/400000 [00:15<00:00, 25411.14it/s]


Το φορτωμένο λεξιλόγιο έχει τις ακόλουθες βασικές λειτουργίες:
* `vocab.stoi` λεξικό μας επιτρέπει να μετατρέπουμε μια λέξη στον δείκτη της στο λεξικό
* `vocab.itos` κάνει το αντίθετο - μετατρέπει αριθμό σε λέξη
* `vocab.vectors` είναι ο πίνακας διανυσμάτων ενσωμάτωσης, οπότε για να πάρουμε την ενσωμάτωση μιας λέξης `s` πρέπει να χρησιμοποιήσουμε `vocab.vectors[vocab.stoi[s]]`

Εδώ είναι ένα παράδειγμα χειρισμού ενσωματώσεων για να επιδείξουμε την εξίσωση **kind-man+woman = queen** (έπρεπε να ρυθμίσω λίγο τον συντελεστή για να δουλέψει):


In [15]:
# get the vector corresponding to kind-man+woman
qvec = vocab.vectors[vocab.stoi['king']]-vocab.vectors[vocab.stoi['man']]+1.3*vocab.vectors[vocab.stoi['woman']]
# find the index of the closest embedding vector 
d = torch.sum((vocab.vectors-qvec)**2,dim=1)
min_idx = torch.argmin(d)
# find the corresponding word
vocab.itos[min_idx]

'queen'

Για να εκπαιδεύσουμε τον ταξινομητή χρησιμοποιώντας αυτές τις ενσωματώσεις, πρέπει πρώτα να κωδικοποιήσουμε το σύνολο δεδομένων μας χρησιμοποιώντας το λεξιλόγιο GloVe:


In [16]:
def offsetify(b):
    # first, compute data tensor from all sequences
    x = [torch.tensor(encode(t[1],voc=vocab)) for t in b] # pass the instance of vocab to encode function!
    # now, compute the offsets by accumulating the tensor of sequence lengths
    o = [0] + [len(t) for t in x]
    o = torch.tensor(o[:-1]).cumsum(dim=0)
    return ( 
        torch.LongTensor([t[0]-1 for t in b]), # labels
        torch.cat(x), # text 
        o
    )

Όπως έχουμε δει παραπάνω, όλες οι ενσωματώσεις διανυσμάτων αποθηκεύονται στον πίνακα `vocab.vectors`. Αυτό κάνει εξαιρετικά εύκολη τη φόρτωση αυτών των βαρών στα βάρη του επιπέδου ενσωμάτωσης χρησιμοποιώντας απλή αντιγραφή:


In [17]:
net = EmbedClassifier(len(vocab),len(vocab.vectors[0]),len(classes))
net.embedding.weight.data = vocab.vectors
net = net.to(device)

Τώρα ας εκπαιδεύσουμε το μοντέλο μας και ας δούμε αν θα έχουμε καλύτερα αποτελέσματα:


In [18]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=16, collate_fn=offsetify, shuffle=True)
train_epoch_emb(net,train_loader, lr=4, epoch_size=25000)

3200: acc=0.6271875
6400: acc=0.68078125
9600: acc=0.7030208333333333
12800: acc=0.71984375
16000: acc=0.7346875
19200: acc=0.7455729166666667
22400: acc=0.7529464285714286


(35.53972978646833, 0.7575175943698017)

Ένας από τους λόγους που δεν βλέπουμε σημαντική αύξηση στην ακρίβεια οφείλεται στο γεγονός ότι μερικές λέξεις από το σύνολο δεδομένων μας λείπουν στο προεκπαιδευμένο λεξιλόγιο GloVe, και ως εκ τούτου αγνοούνται ουσιαστικά. Για να ξεπεράσουμε αυτό το ζήτημα, μπορούμε να εκπαιδεύσουμε τις δικές μας ενσωματώσεις στο σύνολο δεδομένων μας. 


## Περιεκτικές Ενσωματώσεις με Συμφραζόμενα

Μία βασική περιοριστική παράμετρος των παραδοσιακών προεκπαιδευμένων αναπαραστάσεων ενσωματώσεων, όπως το Word2Vec, είναι το πρόβλημα της διάκρισης της σημασίας των λέξεων. Ενώ οι προεκπαιδευμένες ενσωματώσεις μπορούν να αποτυπώσουν κάποια από τη σημασία των λέξεων στο συμφραζόμενο, κάθε πιθανή σημασία μιας λέξης κωδικοποιείται στην ίδια ενσωμάτωση. Αυτό μπορεί να προκαλέσει προβλήματα σε μοντέλα που χρησιμοποιούν αυτές τις ενσωματώσεις, καθώς πολλές λέξεις, όπως η λέξη «play» (παίζω), έχουν διαφορετικές σημασίες ανάλογα με το πλαίσιο χρήσης τους.

Για παράδειγμα, η λέξη «play» στις παρακάτω δύο προτάσεις έχει αρκετά διαφορετική σημασία:
- Πήγα σε μια **παράσταση** στο θέατρο.
- Ο Γιάννης θέλει να **παίξει** με τους φίλους του.

Οι παραπάνω προεκπαιδευμένες ενσωματώσεις αναπαριστούν και τις δύο αυτές σημασίες της λέξης «play» στην ίδια ενσωμάτωση. Για να υπερβούμε αυτόν τον περιορισμό, χρειάζεται να δημιουργήσουμε ενσωματώσεις που βασίζονται στο **γλωσσικό μοντέλο**, το οποίο έχει εκπαιδευτεί σε ένα μεγάλο σώμα κειμένου και *γνωρίζει* πώς οι λέξεις μπορούν να συνδυαστούν σε διαφορετικά συμφραζόμενα. Η συζήτηση για τις περιεκτικές ενσωματώσεις με συμφραζόμενα είναι εκτός του πεδίου αυτής της διδασκαλίας, αλλά θα επανέλθουμε σε αυτές όταν μιλήσουμε για τα γλωσσικά μοντέλα στην επόμενη ενότητα.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Αποποίηση ευθυνών**:
Αυτό το έγγραφο έχει μεταφραστεί χρησιμοποιώντας την υπηρεσία μετάφρασης με τεχνητή νοημοσύνη [Co-op Translator](https://github.com/Azure/co-op-translator). Ενώ επιδιώκουμε την ακρίβεια, παρακαλούμε να έχετε υπόψη ότι οι αυτοματοποιημένες μεταφράσεις ενδέχεται να περιέχουν λάθη ή ανακρίβειες. Το πρωτότυπο έγγραφο στη μητρική του γλώσσα πρέπει να θεωρείται η αυθεντική πηγή. Για κρίσιμες πληροφορίες, συνιστάται επαγγελματική ανθρώπινη μετάφραση. Δεν φέρουμε ευθύνη για τυχόν παρεξηγήσεις ή λανθασμένες ερμηνείες που προκύπτουν από τη χρήση αυτής της μετάφρασης.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
